# 青空文庫から作品をダウンロードして本文を取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/aozora_download.ipynb)

NLP クックブック 01-01 ／ データの収集と入出力

著作権の切れた日本語の文学作品を、手軽に使える日本語コーパスとして取得したい。ルビや注記を取り除きたい。

In [1]:
import io, re, urllib.request, zipfile

def aozora(url):
    """青空文庫の「テキストファイル (ルビあり)」の zip の URL から、本文だけを取り出す"""
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    raw = z.read(z.namelist()[0]).decode("cp932")           # 青空文庫のテキストは Shift_JIS
    header, _, body = re.split(r"-{50,}", raw)[:3]           # 冒頭の「テキスト中に現れる記号について」を除く
    body = body.split("底本：")[0]                            # 末尾の底本情報を除く
    body = re.sub(r"《.+?》", "", body)                       # ルビ: 吾輩《わがはい》
    body = re.sub(r"［＃.+?］", "", body)                     # 入力者注: ［＃「言＋墟のつくり」、第4水準2-88-74］
    body = body.replace("｜", "")                             # ルビの開始位置を示す記号
    return header.strip().split("\r\n")[:2], body.replace("\r\n", "\n").strip()

(title, author), text = aozora("https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip")
print(title, "/", author, "/", len(text), "文字")
print(text[:120])

吾輩は猫である / 夏目漱石 / 321138 文字
一

　吾輩は猫である。名前はまだ無い。
　どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番獰悪


In [2]:
# 生のテキストはこうなっている
with urllib.request.urlopen("https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip") as r:
    raw = zipfile.ZipFile(io.BytesIO(r.read())).read("wagahaiwa_nekodearu.txt").decode("cp932")
start = raw.index("吾輩《わがはい》は猫である。")
print(raw[start:start + 150])

吾輩《わがはい》は猫である。名前はまだ無い。
　どこで生れたかとんと見当《けんとう》がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番｜獰悪《どうあく》な種族であったそうだ。この書


- zip の URL は、各作品の図書カード (`https://www.aozora.gr.jp/cards/人物ID/card作品ID.html`) の「ファイルのダウンロード」欄にある。
- 全作品の一覧は「公開中 作家別作品一覧拡充版」の CSV として配布されている。
- 新字新仮名の作品を選ぶと現代の解析器で扱いやすい。旧字旧仮名の作品は UniDic の近代文語用辞書などを検討する。